# LAB6: TRANSFORMERS INTRO

## Bài 1: Khôi phục Masked Token (Masked Language Modeling)

In [75]:
mask_filler = pipeline("fill-mask")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [76]:
input_sentence = "Hanoi is the <mask> of VietNam"

In [77]:
prediction = mask_filler(input_sentence, top_k = 5)

In [78]:
for pred in prediction : 
    print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

Dự đoán: ' capital' với độ tin cậy: 0.4303
 -> Câu hoàn chỉnh: Hanoi is the capital of VietNam
Dự đoán: ' birthplace' với độ tin cậy: 0.1588
 -> Câu hoàn chỉnh: Hanoi is the birthplace of VietNam
Dự đoán: ' founder' với độ tin cậy: 0.0328
 -> Câu hoàn chỉnh: Hanoi is the founder of VietNam
Dự đoán: ' Republic' với độ tin cậy: 0.0297
 -> Câu hoàn chỉnh: Hanoi is the Republic of VietNam
Dự đoán: ' Voice' với độ tin cậy: 0.0195
 -> Câu hoàn chỉnh: Hanoi is the Voice of VietNam


### Mô hình dự đoán đúng từ capital.
### Những mô hình chỉ có encode phù hợp với tác vụ dự đoán token bị che vì chúng được huấn luyện với khả năng nhìn cả hai chiều để hiểu ngữ cảnh sâu sắc của câu, tức là xem xét cả từ đứng trước và đứng sau để có thể đưa ra dự đoán với xác suất cao nhất.

## Bài 2: Dự đoán từ tiếp theo (Next Token Prediction)

In [79]:
generator = pipeline("text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


In [80]:
prompt = "The best thing about learning NLP is"

In [81]:
generate_text = generator(prompt, max_length = 20, num_return_sequences = 1)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [82]:
print(f"Câu mồi: '{prompt}'")
for text in generate_text:
    print("Văn bản được sinh ra:")
    print(text['generated_text'])

Câu mồi: 'The best thing about learning NLP is'
Văn bản được sinh ra:
The best thing about learning NLP is that I can learn it in two months so if you're not already learning NLP you can just skip it and just learn it in a few months. The other thing is I don't know how to learn it in a year or two so I'd have to ask them.


### Kết quả sinh ra được đánh giá mức độ hợp lý ở mức cơ bản nhưng nội dung vẫn lan man và vòng vo, đôi khi còn hơi lạc nghĩa.  

### Mô hình chỉ có decode phù hợp cho những bài toán sinh từ vì nó được huấn luyện để chỉ có thể nhìn một chiều, có nghĩa dựa vào những từ đã xuất hiện và dự đoán token tiếp theo.

## Bài 3: Tính toán Vector biểu diễn của câu (Sentence Representation)

In [83]:
import torch
from transformers import AutoTokenizer, AutoModel

In [84]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


In [85]:
sentences = ["This is a sample sentence."]

In [86]:
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

In [87]:
with torch.no_grad():
    outputs = model(**inputs)

In [88]:
last_hidden_state = outputs.last_hidden_state

In [89]:
attention_mask = inputs['attention_mask']
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask

In [90]:
print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("\nKích thước của vector:", sentence_embedding.shape)

Vector biểu diễn của câu:
tensor([[-6.3874e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5784e-02,
         -2.1826e-01,  4.7636e-01,  4.8659e-01,  4.0647e-05, -7.4273e-02,
         -7.4740e-02, -4.7635e-01, -1.9773e-01,  2.4824e-01, -1.2162e-01,
          1.6678e-01,  2.1045e-01, -1.4576e-01,  1.2636e-01,  1.8635e-02,
          2.4640e-01,  5.7090e-01, -4.7014e-01,  1.3782e-01,  7.3650e-01,
         -3.3808e-01, -5.0331e-02, -1.6452e-01, -4.3517e-01, -1.2900e-01,
          1.6516e-01,  3.4004e-01, -1.4930e-01,  2.2422e-02, -1.0488e-01,
         -5.1916e-01,  3.2964e-01, -2.2162e-01, -3.4206e-01,  1.1993e-01,
         -7.0148e-01, -2.3126e-01,  1.1224e-01,  1.2550e-01, -2.5191e-01,
         -4.6374e-01, -2.7261e-02, -2.8415e-01, -9.9249e-02, -3.7017e-02,
         -8.9192e-01,  2.5005e-01,  1.5816e-01,  2.2701e-01, -2.8497e-01,
          4.5300e-01,  5.0945e-03, -7.9441e-01, -3.1008e-01, -1.7403e-01,
          4.3029e-01,  1.6816e-01,  1.0590e-01, -4.8987e-01,  3.1856e-01,
          3.

### Vector biểu diễn có 768 chiều, con số 768 tương ứng với tham số hidden_size (kích thước tầng ẩn) của mô hình BERT-base.

### Khi tokenization, mô hình thêm PAD token (padding) vào cuối câu để các câu trong batch có cùng độ dài. PAD token không mang thông tin nội dung, nhưng BERT vẫn sinh ra vector cho nó. Nếu Mean Pooling lấy trung bình toàn bộ token kể cả PAD, thì vector câu sẽ bị sai lệch. Vì vậy phải dùng attention_mask để loại bỏ PAD token và chỉ tính trung bình trên các token thật, giúp embedding chính xác.